# Project 12 — BROKEN notebook (debugging exercise)

This notebook reports a slope that is **biased** and contains a latent-variable **indexing bug** in an attempted EIV fix. Run it, compare the slope to the known truth, find both bugs, and fix them. Answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
x_obs, y, n = data['x_obs'], data['y'], data['n']
print('true beta =', data['truth']['beta'])

### BUG 1 — naive regression on the noisy predictor (attenuation).

This regresses $y$ directly on $x_\text{obs}$, ignoring that the predictor is measured with noise. The slope will be biased toward 0, but nothing in this cell warns you.

In [ ]:
with pm.Model() as naive:
    alpha = pm.Normal('alpha', 0, 5)
    beta = pm.Normal('beta', 0, 5)
    sigma_y = pm.HalfNormal('sigma_y', 2)
    pm.Normal('y', mu=alpha + beta * x_obs, sigma=sigma_y, observed=y)
    idata = pm.sample(draws=800, tune=1000, chains=4, random_seed=RNG,
                      progressbar=False)
print(az.summary(idata, var_names=['beta']))
print('Reported slope is ~1.5 but the TRUTH is 2.0 -> attenuation bias.')

### BUG 2 — an EIV attempt with a latent-variable indexing bug.

Someone tried to fix it with a latent $x^*$, but mis-indexed the latent vector against the observations, so the measurement and structural models no longer line up row-for-row. (Run and read the error / nonsensical fit.)

In [ ]:
tau_x = data['tau_x']
with pm.Model(coords={'obs': np.arange(n)}) as eiv_bug:
    alpha = pm.Normal('alpha', 0, 5)
    beta = pm.Normal('beta', 0, 5)
    sigma_y = pm.HalfNormal('sigma_y', 2)
    mu_x = pm.Normal('mu_x', 0, 5)
    sd_x = pm.HalfNormal('sd_x', 5)
    x_true = pm.Normal('x_true', mu_x, sd_x, dims='obs')
    pm.Normal('x_obs', mu=x_true, sigma=tau_x, observed=x_obs, dims='obs')
    # BUG 2: reversing x_true breaks the row-for-row alignment with y
    pm.Normal('y', mu=alpha + beta * x_true[::-1], sigma=sigma_y, observed=y)
    idata_bug = pm.sample(draws=600, tune=1000, chains=2, target_accept=0.95,
                          random_seed=RNG, progressbar=False)
print(az.summary(idata_bug, var_names=['beta']))
print('Mis-indexed latent -> slope collapses toward 0 (relationship destroyed).')

### The fix — correct EIV with aligned latent x_true.

Use the latent $x^*$ **in the same order** as the observations (no reversal). With $\tau_x$ known, the slope is recovered near 2. See `model.py` (`model='eiv'`).

In [ ]:
from model import fit
idata_fixed = fit(data, model='eiv', draws=800, tune=1500, chains=4,
                  target_accept=0.95, seed=RNG)
print(az.summary(idata_fixed, var_names=['alpha','beta']))
print('true beta =', data['truth']['beta'])